# Cyclistic PySpark Analysis

Full-dataset analysis using the PySpark DataFrame API.

- **Data:** `data/processed/trips_clean.csv` (~14.8 million rows)
- **API:** DataFrame only — `SparkSession`, `df.filter`, `df.groupBy`, `df.select`, `df.show`
- **Timing:** each cell prints its own elapsed time; a summary cell at the end shows total run time

> `data/processed/trips_clean.csv` does not include `rideable_type` or lat/lng columns,
> so the bike-type section is omitted.

In [ ]:
%pip install pyspark==3.5.1 python-dotenv --quiet

In [ ]:
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, to_timestamp, unix_timestamp,
    hour, date_format,
    count, avg, stddev, min, max,
    percentile_approx, when,
    round as spark_round,
)

# Notebook-level timer — starts here, reported in the final cell
_NB_START = time.time()
print("Notebook timer started.")

## Start Spark

In [ ]:
_t = time.time()

import os
os.environ["SPARK_DRIVER_MEMORY"] = "6g"

spark = (
    SparkSession.builder
    .appName("CyclisticAnalysis")
    .master("local[*]")
    # Give the driver enough heap for 14.8M-row aggregations.
    # IMPORTANT: must be set before SparkContext starts — restart the kernel
    # if you need to change this value.
    .config("spark.driver.memory",          "6g")
    # Default shuffle partitions is 200, which is too high for local mode.
    # 8 reduces per-partition memory overhead significantly.
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:          ", spark.version)
print("Driver memory:          ", spark.conf.get("spark.driver.memory"))
print("Shuffle partitions:     ", spark.conf.get("spark.sql.shuffle.partitions"))

print(f"\nCell time: {time.time() - _t:.2f}s")

## Load data

Read the full `trips_clean.csv` (~14.8 million rows).

In [ ]:
_t = time.time()

DATA_PATH = "data/processed/trips_clean.csv"

df_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(DATA_PATH)
)

print("Columns:", df_raw.columns)
df_raw.show(5, truncate=False)

print(f"\nCell time: {time.time() - _t:.2f}s")

## Calculate ride duration

In [ ]:
_t = time.time()

df = (
    df_raw
    .withColumn("started_at", to_timestamp("started_at", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("ended_at",   to_timestamp("ended_at",   "yyyy-MM-dd HH:mm:ss"))
    .withColumn(
        "ride_length_minutes",
        (unix_timestamp("ended_at") - unix_timestamp("started_at")) / 60.0,
    )
)

df.select("ride_id", "started_at", "ended_at", "ride_length_minutes").show(5, truncate=False)

print(f"\nCell time: {time.time() - _t:.2f}s")

## Quality checks

Count missing values, summarise ride length, then filter out rides that are
zero/negative or longer than 24 hours.

In [ ]:
_t = time.time()

# Missing values per column
print("Missing values per column:")
df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
).show()

# Ride length summary
print("Ride length (minutes) summary:")
df.select("ride_length_minutes").describe().show()

# Filter
MAX_DURATION = 24 * 60  # 1 440 minutes

total_before = df.count()
print("Number of rides before filtering:", total_before)

df_clean = df.filter(
    (col("ride_length_minutes") > 0)
    & (col("ride_length_minutes") <= MAX_DURATION)
)

total_after = df_clean.count()
print("Number of rides after filtering:", total_after)

print(f"\nCell time: {time.time() - _t:.2f}s")

## Member vs casual

Compare ride lengths and trip counts between members and casual riders.

In [ ]:
_t = time.time()

# Ride length stats by user type
print("Ride length (minutes) by user type:")
df_clean.groupBy("member_casual").agg(
    count("ride_length_minutes").alias("count"),
    spark_round(avg("ride_length_minutes"),    6).alias("mean"),
    spark_round(stddev("ride_length_minutes"),  6).alias("std"),
    min("ride_length_minutes").alias("min"),
    percentile_approx("ride_length_minutes", 0.25).alias("25%"),
    percentile_approx("ride_length_minutes", 0.50).alias("50%"),
    percentile_approx("ride_length_minutes", 0.75).alias("75%"),
    max("ride_length_minutes").alias("max"),
).show()

# Trip share by user type
total = df_clean.count()
print("Trip share by user type:")
(
    df_clean.groupBy("member_casual")
    .count()
    .withColumn("share_%", spark_round(col("count") / total * 100, 2))
    .orderBy(col("count").desc())
    .show()
)

print(f"\nCell time: {time.time() - _t:.2f}s")

## Time and day patterns

Examine how rides vary by hour of day and day of week for members vs casual riders.

In [ ]:
_t = time.time()

df_clean = (
    df_clean
    .withColumn("hour",        hour("started_at"))
    .withColumn("day_of_week", date_format("started_at", "EEEE"))
    .cache()  # store result in memory so every cell below reads from RAM, not re-processing the CSV
)

# Trigger the cache now so the cost is paid once here, not spread across later cells
_cache_count = df_clean.count()
print(f"Cached {_cache_count:,} rows.")

df_clean.select("started_at", "hour", "day_of_week").show(5, truncate=False)

print(f"\nCell time: {time.time() - _t:.2f}s")

In [ ]:
_t = time.time()

print("Trips by hour of day and user type:")
(
    df_clean
    .groupBy("hour", "member_casual")
    .count()
    .orderBy("hour", "member_casual")
    .show(48)
)

print(f"\nCell time: {time.time() - _t:.2f}s")

In [ ]:
_t = time.time()

DAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Build a sort-index map so days appear Mon–Sun rather than alphabetically
day_index = F.create_map(
    *[v for day in DAY_ORDER for v in (F.lit(day), F.lit(DAY_ORDER.index(day)))]
)

print("Trips by day of week and user type:")
(
    df_clean
    .groupBy("day_of_week", "member_casual")
    .count()
    .withColumn("_sort", day_index[col("day_of_week")])
    .orderBy("_sort", "member_casual")
    .drop("_sort")
    .show(14)
)

print(f"\nCell time: {time.time() - _t:.2f}s")

## Station-based insights

In [ ]:
_t = time.time()

for user_type in ["member", "casual"]:
    print(f"\nTop 10 start stations for {user_type}s:")
    (
        df_clean
        .filter((col("member_casual") == user_type) & col("start_station_name").isNotNull())
        .groupBy("start_station_name")
        .count()
        .orderBy(col("count").desc())
        .show(10, truncate=False)
    )

    print(f"\nTop 10 end stations for {user_type}s:")
    (
        df_clean
        .filter((col("member_casual") == user_type) & col("end_station_name").isNotNull())
        .groupBy("end_station_name")
        .count()
        .orderBy(col("count").desc())
        .show(10, truncate=False)
    )

print(f"\nCell time: {time.time() - _t:.2f}s")

In [ ]:
_t = time.time()

print("Average ride length (minutes) by start station and user type (>= 30 trips):")
(
    df_clean
    .filter(col("start_station_name").isNotNull())
    .groupBy("start_station_name", "member_casual")
    .agg(
        count("ride_id").alias("count"),
        spark_round(avg("ride_length_minutes"), 2).alias("mean"),
    )
    .filter(col("count") >= 30)
    .orderBy(col("mean").desc())
    .show(20, truncate=False)
)

print(f"\nCell time: {time.time() - _t:.2f}s")

## Segment comparisons for business insights

Weekend vs weekday, commute vs leisure, ride length bands.

In [ ]:
_t = time.time()

COMMUTE_HOURS = list(range(6, 10)) + list(range(16, 20))

df_clean = (
    df_clean
    .withColumn("is_weekend",      col("day_of_week").isin(["Saturday", "Sunday"]))
    .withColumn("is_commute_hour", col("hour").isin(COMMUTE_HOURS))
    .withColumn(
        "duration_band",
        when(col("ride_length_minutes") <= 15, "short (<=15m)")
        .when(col("ride_length_minutes") <= 45, "medium (15-45m)")
        .otherwise("long (>45m)"),
    )
)

# Weekend vs weekday
print("Weekend vs weekday trips by user type:")
(
    df_clean
    .groupBy("member_casual", "is_weekend")
    .count()
    .orderBy("member_casual", "is_weekend")
    .show()
)

# Commute vs non-commute hours
print("Commute-hour vs other-hour trips by user type:")
(
    df_clean
    .groupBy("member_casual", "is_commute_hour")
    .count()
    .orderBy("member_casual", "is_commute_hour")
    .show()
)

# Ride length band distribution (% of trips per user type)
band_counts = df_clean.groupBy("member_casual", "duration_band").count()
totals      = (
    df_clean.groupBy("member_casual")
    .count()
    .withColumnRenamed("count", "total")
)

print("Ride length bands (% of trips) by user type:")
(
    band_counts
    .join(totals, "member_casual")
    .withColumn("pct_%", spark_round(col("count") / col("total") * 100, 1))
    .select("member_casual", "duration_band", "pct_%")
    .orderBy("member_casual", "duration_band")
    .show()
)

print(f"\nCell time: {time.time() - _t:.2f}s")

## Total run time

In [ ]:
_total = time.time() - _NB_START
_mins  = int(_total // 60)
_secs  = _total % 60
print(f"Total notebook time: {_mins}m {_secs:.2f}s  ({_total:.2f}s)")